# STCA-LLM Stage 3


In [ ]:
import os
import sys
import copy
import pickle
import random
import argparse
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import StepLR
from einops import rearrange
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from transformers.models.gpt2.modeling_gpt2 import GPT2Model
import matplotlib.pyplot as plt
from tqdm import tqdm
import scipy.sparse as sp
from itertools import chain


In [ ]:
def setup_seed(seed):
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True

setup_seed(438)


In [ ]:
class Args:
    epochs = 20
    input_size = 20
    seq_len = 10
    output_size = 1
    lr = 0.0001
    batch_size = 64
    optimizer = 'adam'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    weight_decay = 1e-4
    step_size = 150
    gamma = 0.5
    is_gpt = 1
    gpt_layers = 6
    d_model = 768
    dropout = 0.1

args = Args()
device = args.device


In [ ]:
class Attention(nn.Module):
    def __init__(self, dim, dropout):
        super().__init__()
        self.dim = dim
        self.scale = dim ** -0.5
        self.q = nn.Linear(dim, self.dim, bias=False)
        self.k = nn.Linear(dim, self.dim, bias=False)
        self.v = nn.Linear(dim, self.dim, bias=False)
        self.kv = nn.Linear(dim, self.dim*2, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, y):
        B, _, _ = x.shape
        q = self.q(x)
        kv = self.kv(y)
        kv = rearrange(kv, 'b n (e d) -> e b n d', e=2)
        k, v = kv[0], kv[1]
        q = q*self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(dim=-1)
        out = attn @ v
        return self.dropout(out)

class selfAttention(nn.Module):
    def __init__(self, dim, heads=16, dropout=0.):
        super().__init__()
        self.dim = dim
        self.heads = heads
        head_dim = dim // self.heads
        all_head_dim = head_dim * self.heads
        self.scale = self.dim ** -0.5
        self.q = nn.Linear(dim, all_head_dim, bias=False)
        self.k = nn.Linear(dim, all_head_dim, bias=False)
        self.v = nn.Linear(dim, all_head_dim, bias=False)
        self.kv = nn.Linear(dim, all_head_dim*2, bias=False)
        self.qkv = nn.Linear(dim, all_head_dim*3, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, _, _ = x.shape
        qkv = self.qkv(x)
        qkv = rearrange(qkv, 'b n (e d) -> e b n d', e=3)
        q, k, v = qkv[0], qkv[1], qkv[2]
        q = q*self.scale
        attn = (q @ k.transpose(-2, -1))
        attn = attn.softmax(dim=-1)
        out = attn @ v
        return out


In [ ]:
class GraphAttentionLayer(nn.Module):
    def __init__(
        self,
        in_features: int,
        out_features: int,
        n_heads: int,
        is_concat: bool = True,
        dropout: float = 0.1,
        leaky_relu_negative_slope: float = 0.2,
    ):
        super().__init__()
        self.is_concat = is_concat
        self.n_heads = n_heads
        if is_concat:
            assert out_features % n_heads == 0
            self.n_hidden = out_features // n_heads
        else:
            self.n_hidden = out_features
        self.proj = nn.Linear(in_features, self.n_hidden * n_heads, bias=False)
        self.proj_attn = nn.Linear(in_features, self.n_hidden * n_heads // 2, bias=False)
        self.attn = nn.Linear(self.n_hidden, 1, bias=False)
        self.activation = nn.LeakyReLU(negative_slope=leaky_relu_negative_slope)
        self.softmax = nn.Softmax(dim=2)
        self.dropout = nn.Dropout(dropout)
    def forward(self, h: torch.Tensor, adj_mat: torch.Tensor):
        batch_size, n_nodes = h.shape[0:2]
        g = self.proj(h).view(batch_size, n_nodes, self.n_heads, self.n_hidden)
        ga = self.proj_attn(h).view(batch_size, n_nodes, self.n_heads, self.n_hidden // 2)
        g_repeat = ga.repeat(1, n_nodes, 1, 1)
        g_repeat_interleave = ga.repeat_interleave(n_nodes, dim=1)
        g_concat = torch.cat([g_repeat_interleave, g_repeat], dim=-1)
        g_concat = g_concat.view(
            batch_size, n_nodes, n_nodes, self.n_heads, self.n_hidden
        )
        e = self.activation(self.attn(g_concat))
        e = e.squeeze(-1)
        e = e.masked_fill(adj_mat < 1, float('-inf'))
        a = self.softmax(e)
        a = self.dropout(a)
        attn_res = torch.einsum('bijh,bjhf->bihf', a, g)
        if self.is_concat:
            return attn_res.reshape(batch_size, n_nodes, self.n_heads * self.n_hidden)
        else:
            return attn_res.mean(dim=2)


In [ ]:
class GPT4TS(nn.Module):
    def __init__(self, device='cuda:0', gpt_layers=6):
        super(GPT4TS, self).__init__()
        gpt_weights_path = '../gptweights/gpt2_weights_stage2'  # Adjusted path
        if not os.path.exists(gpt_weights_path):
            gpt_weights_path = 'gpt2' # Fallback to huggingface
        self.gpt2 = GPT2Model.from_pretrained(
            gpt_weights_path, output_attentions=True, output_hidden_states=True
        )
        self.gpt2.h = self.gpt2.h[:gpt_layers]
        self.U = 2
        for i, (name, param) in enumerate(self.gpt2.named_parameters()):
            if 'mlp' in name:
                param.requires_grad = False
            else:
                param.requires_grad = True
    def forward(self, x):
        return self.gpt2(inputs_embeds=x).last_hidden_state

class stcllm(nn.Module):
    def __init__(self, configs, device, adj):
        super(stcllm, self).__init__()
        self.device = device
        self.input_len = configs.seq_len
        self.input_dim = configs.input_size
        self.output_len = configs.output_size
        self.d_model = configs.d_model
        self.adj = adj
        self.dropout = configs.dropout
        self.adj = torch.Tensor(self.adj).to(self.device)
        self.gpt = GPT4TS(device=self.device, gpt_layers=6)
        self.gat = GraphAttentionLayer(
            in_features = self.d_model,
            out_features = self.d_model,
            n_heads = 32,
            is_concat = True,
            dropout = self.dropout,
        )
        self.attention = Attention(dim=self.d_model, dropout=self.dropout)
        self.layer_norm = nn.LayerNorm(self.d_model)
        self.fc = nn.Linear(self.d_model, self.output_len)
        self.proj2 = nn.Linear(self.input_len, self.d_model)
        self.conv = nn.Conv1d(self.input_len, self.d_model, kernel_size=1)
        self.layer_norm1 = nn.LayerNorm(self.d_model)
        self.layer_norm2 = nn.LayerNorm(self.d_model)
        self.layer_norm3 = nn.LayerNorm(self.d_model)
        self.layer_norm4 = nn.LayerNorm(self.d_model)
        self.dropout = nn.Dropout(self.dropout)
    def forward(self, x):
        means = x.mean(dim=1, keepdim=True).detach()
        x = x - means
        stdev = torch.sqrt(torch.var(x, dim=1, keepdim=True, unbiased=False) + 1e-5).detach()
        x = x / stdev
        x = x.permute(0, 2, 1)
        x_time = self.conv(x.permute(0, 2, 1)).permute(0, 2, 1)
        x_time = self.layer_norm1(x_time)
        x_space = self.proj2(x)
        x_space = self.gat(x_space, self.adj.unsqueeze(-1))
        x_space = self.layer_norm2(x_space)
        x_s_to_t = self.attention(x_time, x_space)
        x_t_to_s = self.attention(x_space, x_time)
        x_s_to_t = self.layer_norm3(x_s_to_t + x_space)
        x_t_to_s = self.layer_norm4(x_t_to_s + x_time)
        output = self.layer_norm(x_s_to_t + x_t_to_s)
        output = self.gpt(output)
        output = self.fc(output)
        output = rearrange(output, 'b m l -> b l m')
        output = output * stdev + means
        output = output.permute(0, 2, 1)
        return output


In [ ]:
class MyDataset(Dataset):
    def __init__(self, data):
        self.data = data
    def __getitem__(self, item):
        return self.data[item]
    def __len__(self):
        return len(self.data)

def nn_seq(num_nodes, seq_len, B, pred_step_size):
    data = pd.read_csv('./dataset/wind speed of 20 turbines.csv', header=None)
    train = data[:int(len(data) * 0.6)]
    val = data[int(len(data) * 0.6):int(len(data) * 0.8)]
    test = data[int(len(data) * 0.8):len(data)]
    scaler = MinMaxScaler()
    train = scaler.fit_transform(data[:int(len(data) * 0.8)].values)
    val = scaler.transform(val.values)
    test = scaler.transform(test.values)
    def process(dataset, batch_size, step_size, shuffle):
        dataset = dataset.tolist()
        seq = []
        for i in tqdm(range(0, len(dataset) - seq_len - pred_step_size, step_size)):
            train_seq = []
            for j in range(i, i + seq_len):
                x = []
                for c in range(len(dataset[0])):  
                    x.append(dataset[j][c])
                train_seq.append(x)
            train_labels = []
            for j in range(len(dataset[0])):
                train_label = []
                for k in range(i + seq_len, i + seq_len + pred_step_size):
                    train_label.append(dataset[k][j])
                train_labels.append(train_label)
            train_seq = torch.FloatTensor(train_seq)
            train_labels = torch.FloatTensor(train_labels)
            seq.append((train_seq, train_labels))
        seq = MyDataset(seq)
        seq = DataLoader(dataset=seq, batch_size=batch_size, shuffle=shuffle, num_workers=0, drop_last=False)
        return seq
    Dtr = process(train, B, step_size=1, shuffle=True)
    Val = process(val, B, step_size=1, shuffle=True)
    Dte = process(test, B, step_size=pred_step_size, shuffle=False)
    return Dtr, Val, Dte, scaler


In [ ]:
def get_val_loss(args, model, Val):
    model.eval()
    loss_function = nn.MSELoss().to(args.device)
    val_loss = []
    for seq, labels in Val:
        seq = seq.to(args.device)
        labels = labels.to(args.device)
        preds = model(seq)
        preds = preds[:, :, :1]
        preds = preds.mean(dim=-1, keepdim=True)
        total_loss = 0
        batch_size = seq.shape[0]
        for k in range(args.input_size):
            total_loss += loss_function(preds[:, k, :], labels[:, k, :])
        total_loss /= batch_size
        val_loss.append(total_loss.item())
    return np.mean(val_loss)

def train(args, Dtr, Val):
    adj = pd.read_csv('./dataset/DTW_normalized.csv', header=None)
    adj = np.array(adj)
    model = stcllm(args, device, adj).to(device)
    # Load pretrained from Stage 2 model if available, or just scratch if not?
    # The original code loaded from STCA-LLM_stage2/models/gpt.pkl
    if os.path.exists('../STCA-LLM_stage2/models/gpt.pkl'):
        model.load_state_dict(torch.load('../STCA-LLM_stage2/models/gpt.pkl')['model'])
    loss_function = nn.MSELoss().to(device)
    if args.optimizer == 'adam':
        optimizer = torch.optim.Adam(model.parameters(), lr=args.lr, weight_decay=args.weight_decay)
    else:
        optimizer = torch.optim.SGD(model.parameters(), lr=args.lr, momentum=0.9, weight_decay=args.weight_decay)
    scheduler = StepLR(optimizer, step_size=args.step_size, gamma=args.gamma)
    min_epochs = 2
    min_val_loss = 5
    if not os.path.exists('./models'):
        os.makedirs('./models', exist_ok=True)
    for epoch in tqdm(range(args.epochs)):
        train_loss = []
        for (seq, labels) in Dtr:
            seq = seq.to(device)
            labels = labels.to(device)
            preds = model(seq)
            total_loss = 0
            for k in range(args.input_size):
                total_loss = total_loss + loss_function(preds[:, k, :], labels[:, k, :])
            total_loss = total_loss / preds.shape[0]
            optimizer.zero_grad()
            total_loss.backward()
            optimizer.step()
            train_loss.append(total_loss.item())
        scheduler.step()
        val_loss = get_val_loss(args, model, Val)
        if epoch + 1 >= min_epochs and val_loss < min_val_loss:
            min_val_loss = val_loss
            torch.save({'model': model.state_dict()}, './models/gpt.pkl')
        print('epoch {:03d} train_loss {:.8f} val_loss {:.8f}'.format(epoch, np.mean(train_loss), val_loss))
        model.train()
    torch.save({'model': model.state_dict()}, './models/gpt.pkl')

def test(args, Dte, scaler):
    print('loading models...')
    adj = pd.read_csv('./dataset/DTW_normalized.csv', header=None)
    adj = np.array(adj)
    model = stcllm(args, device, adj).to(device)
    model.load_state_dict(torch.load('./models/gpt.pkl')['model'])
    model.eval()
    print('predicting...')
    ys = [[] for _ in range(args.input_size)]
    preds = [[] for _ in range(args.input_size)]
    for (seq, targets) in tqdm(Dte):
        targets = np.array(targets.data.tolist())
        for i in range(args.input_size):
            target = targets[:, i, :]
            target = list(chain.from_iterable(target))
            ys[i].extend(target)
        seq = seq.to(device)
        with torch.no_grad():
            _pred = model(seq)
            _pred = rearrange(_pred, 'b m l -> m b l')
            for i in range(_pred.shape[0]):
                pred = _pred[i]
                pred = list(chain.from_iterable(pred.data.tolist()))
                preds[i].extend(pred)
    ys = np.array(ys).T
    preds = np.array(preds).T
    ys = scaler.inverse_transform(ys).T
    preds = scaler.inverse_transform(preds).T
    mses, rmses, maes, r2s = [], [], [], []
    for ind, (y, pred) in enumerate(zip(ys, preds), 0):
        print('--------------------------------')
        print('wind turbine:', str(ind+1))
        mse = mean_squared_error(y, pred)
        rmse = np.sqrt(mse)
        mae = mean_absolute_error(y, pred)
        r2 = r2_score(y, pred)
        print('mse:', mse)
        print('rmse:', rmse)
        print('mae:', mae)
        print('r2:', r2)
        mses.append(mse)
        rmses.append(rmse)
        maes.append(mae)
        r2s.append(r2)
        print('--------------------------------')
    if not os.path.exists('./results'):
        os.makedirs('./results', exist_ok=True)
    df = pd.DataFrame({'mse': mses, 'rmse': rmses, 'mae': maes, 'r2': r2s})
    df.to_csv('./results/result.csv', index=False)
    plt.show()


In [ ]:
Dtr, Val, Dte, scaler = nn_seq(args.input_size, args.seq_len, args.batch_size, args.output_size)
print(len(Dtr), len(Val), len(Dte))
# train(args, Dtr, Val) # Uncomment to train
test(args, Dte, scaler)
